# Native FlashVSR+ for two T4s — TeamAIQ engine

This replaces the slow SeedVR2 path with [TeamAIQ's dedicated FlashVSR+ upscaler](https://github.com/TeamAIQ/Colab-notebooks/blob/237383fb084b671c1eb5224db822d88fca6fc8fa/kaggle_notebooks/flashvsr-video-upscaler.ipynb). TeamAIQ/AIQUEST credit and native Gradio interface are retained. Their **inference functions/classes are unchanged**, checked by AST identity before launch. Source and model revisions are pinned and verified.

1. First run **JJP_FlashVSR_Models.ipynb** on **CPU / Accelerator None**. It automatically submits one private Dataset using the working camera-model uploader. Wait for processing. These are **different weights**: your old SeedVR2/Real-ESRGAN Dataset cannot substitute.
2. Attach that Dataset here. Select **GPU T4 ×2**, Internet ON. Stop the older H3/SeedVR2 studio. Run this one cell.
3. Wait for **Engine ready** and the native **Cloudflare/Gradio** links. Initial model load and a small upstream warm-up happen once. No uploaded clip runs automatically.
4. In **Input video**, upload your clip. Leave **Turbo / 2x upscale / Normal motion**, then press **Upscale Video**. Progress reports chunks, elapsed time, and measured per-GPU timings.
5. Completed MP4s live under `/kaggle/working/jjp-flashvsr/outputs/`. Preserve notebook output files before ending the session.

The native engine pipelines chunks across both GPUs with resident FP16 weights, FP32 attention accumulation/norm safeguards, a compact decoder, and streamed output. It deliberately skips slow Triton attention on T4 and chooses a tested PyTorch attention backend on the actual devices. It does not use SeedVR2's BF16 VAE or all-block CPU swapping.

Initial UI changes only: **2×**, **CRF 14**, **veryfast** encoder, comparison copy **OFF**. Turbo favours speed; Base is available for a finer-detail comparison. Colour correction remains the native optional setting. No arbitrary temporal downsampling or reduced model precision has been added.

**Budget protection:** the parent stops only this owned engine if startup exceeds 6 minutes or a clip exceeds the editable 8-minute limit. Those are spending limits, **not measured render-time promises**. Partial work is not a completed video; already completed clips are preserved. The engine's native Stop button can stop earlier.

This is a new backend, not a claim that SeedVR2 was made fast. Full-size T4 speed, fit and restoration quality have not been measured here. Local tests cover source identity, native CPU attention, media/output helpers and process/budget controls. No new user GPU session is required for those tests. AIQuest H3/Qwen camera and existing notebooks are unchanged. Nothing is pushed to GitHub by this notebook.


In [ ]:
MAX_JOB_SECONDS = 480
MAX_STARTUP_SECONDS = 360
from pathlib import Path
import base64,io,zipfile,sys,importlib
WORK=Path('/kaggle/working/jjp-flashvsr')
RUNTIME=WORK/'runtime'
RUNTIME.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('UEsDBBQAAAAIAAAAcV0jewmnbwsAABciAAAPAAAAZmxhc2hfY29tbW9uLnB5nVpbb9w2Fn73r2CzwEqKZWXiZIOFs1OgbdIgzRVO0n1wjQEtcWZYS6IqUrZnDf/3PeeQ1G00Y6cCUo8kkuf+nYv66NGjn1VTZiKLmTa8zHidHeXyoub15kiV+YYtZS4YvGBVrVKhNVOVqLmRqtRsqWpm1oJdSpM8evToYFmrglXcrOEEJotK1YZ9htsD93vNNb7yt39qVfrfSvtfet0Ymbd3clXy7q65cGz4J0YWwv9u6hxOT2rxVyO0OTh48/ZnNmdPZ8fPHz9+dnBwkIklnM6P//UiRCajkwMGVyZXsBoWOu4StySit9fSrFHkkrbELKgvgohxDdqqBS/sEXihLi5ylV4yWTJpRB3mvLjI+IlbCWzxLPz3Y8vOcRSziyCIuv0dK0lTZdyIkE6zXNTCNHXp36/Fjf0FPFqprnguM8ch11oYdzA+AcnQBlbk/mn4IJF6gRYOI2tjfAR+ACfDn4WW/xNsPrdHngUXGyN0cE4r+3rsLbGPg3PHFzeqkOkCDe24A04bsZc74qHitShNUlxmsg7tjZ5/rRsRM3EjgTV1Sbd2ixHoAOCycBptR6stdLNcypswwMOM5HkwWpxc12CnhRE3JkQOk6wpKh0ShzFYMQOi8+OYgSC8yc0c7BiNj6hFlfNUOPadixlVhc5NnaRy6cMnqVSeg7alZqUy7KMqRecD1jCWRL3pniudXMo8r1Zhe4rEgKXYSL68ffP19ekHy5q4SUUFUWfXvVfqsqle17Wq76fiz77m0oQYV6ox86ezwbldACZf7YrXN5WsRdYdMzjzQdy/e/v+fdRueYAEIyn2Mm9tcmFBLkxVUYD7xixXq5i5dTFLr7M5mgK8q7yiX85wgGpfAeCsByJCbjRQ1hWgn7wSTFyJEo3LWbqWeWYBQCMCcJap6zJXPGMADASiCAAEk3gwHFUbkYHDIhNJocAbVCnTLdghRgMeAF6AP4talqv500kEcjhDXh0Gf5S/fPrw4aePr05YcAhuCngEcRMWvAphIYhsNRFF0SGsDaKxJoGznrE/W16mT7D6g39WffAvHphqx6VNhkaybGP6yURdz3tEv3x99enb19jqalGK64WGx5B5eqE/6XHXa9T3dsQNo611uJxXetIUR85KW1vA5H7Xj96Ltg8mN+VSC+ZjBd04XAZfDF8JcnQBTslu3Ql3OmHv1eqE3YLR73om6UwjS9Pur5uyBG+A5Y6Xk2S2vNMv0bf9GTFb5o1ejxS2U3EdoYloKmQZPpvFrOA34Sx52obPkSMfRdsUHg4bA/KQTQYvevhp4z5V2YQlx1Hjc7VAvS9QLZPk2teJFuIyJPlim5bgYTRKiEcvZrPZhKikUC5z8KPuQEr6UZIJZDik53oeuKwxYWHrLqdNibodu4tEwLjdVsRd323+KG+Ri777LCUAbT4y9SBLOZQ0NS81YExI2RwMzOsV/gXEE/WVmENB1eHiqdBNwS8g0K4AlZYSePOQ95KlQFDUjGeZxFIRyQM2QI0E5AAbr+HBEYrIclm44tHqDwn6msDeRb0331kXgNu4ffRSh716S/qqyQvZr5tac9hKyS7pEui2kYLXSACi0UL9EiwA+kjXAtJBU7wE77UqzBCNAfG8bL7koQIFEciy269f3CN7dziuZrDs5AYLEoNpBwy4EuGznhhqudSkUrdvXN5haLk3XkmAbSDfrK8qf8pkLeirTzplWpGehK+X+qbdr+yO+I9j4iNbTVjlbUm8tQpe2rwwsoV724tpuXRtSJJJfbloNARgOHDBKFnWQrD/jFg6srweOhL3c/gRikBRqma1ZkiK7BkcOlIlL3qo7foaMOWw0UlO7d/Q1+CqgTALzmO2BvQRADi3wSn6RXDClpbR+a3l8+4ouOtpmOx+e7cvsyLCjsjDLWGuu2/zwvzFjNDX1UsTeF0jhECH0VqYzcih/A7y1Uaj2x3PXkxlYXcCbsJyut3oJE9Ai2HwiyoNmOzIKgHyQkCYXhuN4oROJ6zTSbQvlw/t960UN5VITQ/9bBBOgDtWZsBIJ28nwCjiBur2sUNKDvhFMN4XXLs8lwnEIGrOpyXY3aS2mntAm9q/eiRd5dlrW6cur4TDOctBnntWg6R+w497o398TdnqlTdQW3lVsizhD8ECouHIaEAdveoB+HYvSZ8KWCF1wU26PtkV5nj9fbh0BdenLxMtEzYqLlVgRO2QwNUC6VopLRY6rZFbCm4s2KAYUMpoqCtVDhGGtQGgI1QB8zDqaoOPCrUEDpqxd3y1Au3+1SjDE/YROqaa6Qq9jF0LuVobaGRA/aWBCoGntYLOo4Bmzei2KEghuiVORLAnOTtv0x4yQjkPGeq1t/jYVRD4e4Dq+KDLcwgb0J3yFIsgWhzj/X8Xn96N7EtgP59ICkSBcsFgfcdzwiuI2ywMcY3VXtSl7u4hSqIVdhthtzlml2IztzEKq65P8D9ns3NUOyhSC1vsdLz+g52CPQTQr3gqzYbZGR5QEjc8NQxdzfo79BpvARZXNa6C3ogcFN5dbGik5xpm7L1WWNZQf5u0dFJwBWmoRyxCqiPIItAJGOiLY2ZLi7L1DtQ96eqJ9ZsnfmnUTaCm4xlttGPn9qAK/3bnlDbI59SxPH8M1WvMvB8fWQn6ba/rrazDdxZkt0j9DsprNNYTOOUkOV7esTfyZzLfS6LDbi214Xu+BIztlABY54oQvaMzAzWRq0GtY88bumFbH/cUMg5xByqgVr0pcllehg/Eqk/XiIUu4O1wrmig3kAIvIAcy9x5kyXskENXoE+U5C0DW+g12foAkFilebZIeRsNKIbzWrCNs+e2XRIW7MoQeOybz98AaDJIoIBDNdoGKoYsgfih0gBHOrngEJN25IBTHAwsjm03dS6Um13pZnlEWHi5hyqAL4cwy9grbjjmf+jBamzt3DAQMx35iRM2CbqRIixbWHYBHgyn/laWVWMWFrZSPLkD4F9UtQF+aqhMIP7tPATn2xDK6K8IwiAHnADdnihTCF8c3fOWM4LgFoG3CHl87V5ErmG2fDjchp8PbdUoqSLAO+nOAvsksHiPuIVvb/lZgL+h86DOh0ZttPDOcZqJG1x310Ks1YKqNzFbxNZ/cJfC6UZ+2RdhOMxHMgTKUELSroj90/IxyqpIMoFVbk4c4poYElXkoZ8U07IRPenyvS3VAeahKtAIs8h53E9yJFwnZW/M6qGA1PzE1UbIqFPZEA62G2D3vitw9jXCOwI0eE+OT465XeLc1/b6y2nhzMlg7XvedsNdbZOPmwgiwxArb0wYVqSximpbMgpW/4NDySzU8t77lQMcmubAY31ZkjuHic6Q3vQD6sOztqXZqwui66cajkqv6JmwzU8EN5TNP318zTQHO7DffvvMfs0BCX7/cso+WPxzQZ+wD84Lyd1P/igDGg0nfypZho6kn1iAcNZfsQDg3TxgFJXUSXuROnF+aN3Wu2w7sNkusSyk+H7bUT6kfL5fAb3eugNoZpSFU8bz3GeAClIxsP2ckofzWShWB1nCJYj+3GV/bMZO8ukYjXeYejCDcPt9JIxmKlSxBL/76RvJeBLEbOTyuybAKbTFsmzEiO3vnLERJ39/foWX/ZgZ9/rj8afYuNcbO7G/uMIU60OX2R4oOyVuq1rXU/s5sSxTVcBB8ajn9v21asxKDeLOu8J0U+3Pe2hT7c/f30nv/Ercvx7WZn93ez09ZIMMCuiPU4+MZU1N/QIUIAT6Vs/DFrdH9oetiSKl7PF37t46/4n5/tmaL2b6I1k7q05pYPGyddwHjAT3teT9z9w21AM/E0/wYRD7sgYLOpB+sUCFLBYY2sFiUXBZLhaB+1bs/leHjW4hcQNdZL26Ont6ThsIvoL+h79edUhfshGrtK072s3HUJCgH9oP3vjVrn317Lx38xyw+P9QSwMEFAAAAAgAAABxXZHoafkuAwAAiQcAABYAAABmbGFzaF9jaGVja19kYXRhc2V0LnB5nVVNj9s2EL3rVwzcg+Ril9hFDw021SFIE7QoEiyS5mQsBJoa2YQpUiEpr40g/z1DUh92nO0W0UUkNR9v3huOFovF321nrAe/ReDC91zB6/tP0HfK8Bot1NihrlELiQ7W2BhLdvoIralRgbdcuwYtWywWmUyR0kvJNWvR85p7Pn3RrkPhx63r1501Ap2bTo4uy7IaGxBbFLvK9trLFovlXQb0jFbyELeNNS3s+GajkPFOsrSsaFnhwQfU9ejyT/z0qpMXjrEOx1K9VSMVjj6f4tFbOolOv8CbA1ohHUauNqjRck8pBia23AMeOuPoKBj0lF8obmlLgGcejyyG487hWZLCmx3qMp8Nr4mdNeZL5k1VS+GL5SqPRvkDlCVcWsbAqRIoR7aZkxvNfW+xmFg4LdctWcctJ63QuhhBWKTCnokQdHXoq2RcaXy8iCMb0MbDl7wxilopv4Lc4ucenc+/wh8lkHuRgAz6hsfywPCHJPwba40t8pQVCIPc49yaEyyoZUNN6JKwgXyLe4mPRP2ejqXRLF8+Banr10qKsBJGk7mviG7h9uGklrYK8s54U7n/D29n5T4Q+WeiKvFKYH4G+LB1pMqXr6mJjSVGWmpXDUWeujlgTitX78KGOi8/Aevtcd6chl2FSA9B8ovLywabIpgsJ288COx+dNnZPReEAd8b/9bQJYicPJM1D6JQt3muVLgvGDrJozqmliYitS/y+1cfP95NbA6zYUDgAg2BQIqEa2N2cH/0W6PviIUx3RU0qnfb8l/bD4VYJBX0yXRIw6ezAQB+P35oxr3jXmzhr9/gPU2YO9ghdvBo7E7qzZy5SwS4l2NJYLQ6wuMWNbTSOTKO8/JCkQHOd7MvO6H7nal7dU4tTRLAc46HJERsEb+w2CdkmtMwcZ2SRCY11urmYXKhizF5lalvzjUbi6Kgq/C5LG/Z7e/sJp9joDqJ8mRXLv8j7mz/VAaH5+7x+k0n8x+FEX3Fiv4nDA8oes/XCinsdRuvvOzCa1AnLK+vP/cSPS1/HQE9XCUhYrtcQdDC9L68fXGzfE6wLCMiqiqwXlWRz6pqudRVNZB60WDZN1BLAwQUAAAACAAAAHFd9qySGrIJAABJGwAAEAAAAGZsYXNoX2RhdGFzZXQucHnVWN1227gRvtdToM4FqViim3S359SpLpREyXqd2D6Wvdutjw8PREISIpLgAqBkberbfYA+Yp+kM/ghqR9b6d5VFzYBDIDB4JtvZnB0dHRZMFJKvqSakfdUU8V0j1SKFzOi54ywB640Ns5yOmNkrKuUC1JQzZeMKC0ZzWG0X5WZoCnJmZ6LNDo6OupMpchJSfU84xPC81JITa6g2XHf06TQmW/MqUI53+SFKlmiffOLEoX/Fsp/Sea/NMvLKc+Y3XOawWJxIvJcFH7jj2dve4RqkfMkxtV6JJkLoVisEkl1Mu8RLWmhpkz2yJJmPO10OimbElqBDQrNE7BOGtOSh93TDoHfC/JBZJlYGRuthFzsmCiRLMWpNAONxCoiF2zJJJq60CdKC8naIrBpxVRk1uZTUgg8acSKJZeiiGZMh8H58OPHT6N4eHUW31yejy6CLqFF+pzo+eiXwOmLPy3XTQN/xlwLOptlYAgG2mjlDXarmBzbrncZByU3Jjb73e2qdU8Gu9PDLmrmdtl3lnp99pCwUpOR+cdFsalySZX6PzGSF95rjUPmvB2Pri+Gn0fGmm6lfRas5Q6uiOc8sJgxxf96ES/INVMiAzY4N0YiY2+kwjhHSSWcGG+ATYRYnBIOY7lIKxDlytyMkYRvDuugoxHQ3bpC26HnLFnEqWUofwOlZLh+LCtwo5yZOVaNYcnhsFvjzu7UjNVyTW/U9nfXDWepZIGjjhPgdDDqNQlBlGZi1iOi0mUF3MkL+BdLIeA7oaA2DK0KJgdB0MNlBheiYA5xT84B/ZAsQyvQ7dlWI+Z7jHC3tVaUL1IuQ2t1NbiRFexvSDwWC9O00mbitwpnIlmARqJkhdPoJHDnj3AMDhasHHQMr0dT7A7xT8/1fLp8dx6P/kH+1W5fvHW6Sz7jBfDggKB1OjueaK8M/wq5j5URh8NWN5mwKVJsCqbHyIT0TIs1Ao9lUb2suRg8mfmPSwMIElFM+Sx2lGy4ogIPLmjO8KBAKSD4BJvU/mgE630cW0kWTassyzHkhDK4G/b/Sfu//bn/t7h/fxw4nHQ3HU1Srhi5tgAeSSlkGIzBA94Pb4bj0U18+fPF6JpoQdaikt4Hvb5vyIKxshVojFtuOmrU0nMKdmLSxCiwiovKkZrT19//NcTAGaVVXqoG9Aq8MF6wtQVPF2ySgIXDbjeas4eUz5gCsrs7ffX6vt5jDoScMW/04+Dky5eyb3x8qWQ/OG7pUM+pJEIjmGtdqtOTk9VqFVlKhsvKTxwU1UlwbBev50mWMF7iWbZg6xKWCM8UtK/JzYiMF6jQhg9zcpRXoR8HEkhjzR40ntUgwG4N6BgM3Bm3GBNPFAZXki25qFS2Jqqa5Fwj4W0lYKeABTgyuE5WqXnLEZtzGVICmQbKjQshiO35YkyLVC2j+Awyt0qi9V2OFdV9oV9hB7Zfg6nIUiYR1JL9Cl6hg0fyd4wlOqznR0AjADnNpPoGCJ8VCiCUwdk9ZI3C4IWNkimfQkambBTAWIKmYyuY4/JPl24CZ9SOriAppRJs2oa1AtdIGIbir48N1tHdlQkmBXGAvgtMjwrut5IAkE3gTnmKtwTyDRNHcpaJSWjm3QWGJO67aLmWSA0mloEhwi3zOEubvDOsd+lZ5fbItk50t7Ethvd6PiDUhOWtXMP/JgDgReNf1vhAphkC6AMQBevUoxj1bKofW8HwJZUz1SMvXy5W+LWlZCHgKhKAYnvZLfwaFGE6UsNnwovdhSP4W+UYne5q6N13NtYy2jGYa0G6x2ATUYETH9xr/7y2Bs4NTBqlpd9wZ57zTu9P4ct6IbOdbflNN2Z7g8EGd/c7CPxWtOLPAgQVtUgxHLUJ0p05SMhpPQWzszoYt3+AVSP5HDJhFV9R2V1dXnNyQAXHONYX7FLPO8I+bvmMQZ6YhFFVOcm5MhHX5wTWyKckOD6gzIrreV1cRjcMc04q1+85BAEo4NYhJE8Dn8JBqjnlD4MgApP1cUIfaw8FJOak9+v/gtwAsY3fnzu1FKGJrsB3JmsNjKXnUlSzuWG/jBcLX0a6ehu7kAUphjVMS37jJVkxPptrFe3dLzRpY61Ud+tCIrXOcdFYC2f+hkf2E0kCaEZHziA3dB4amaizX9xFwdvSZWUY6jZUeDro+Z9x9uYEe2UAR7sKkT8NrLrHr/ZfBv72AerChps6RAHDunzO1gYE14ccbMGKb4CVd/KIliUe5YB469miTr7tEv1Sihlcj7JpTA8CtUtCTl0GAgHb7wZ9/hN6XRoU22IGBw3nP+71yIPeHlUFgsam4ZcFpDamnjOLCxlATK4S4CCFye+6ITntPaqO3puY/WP+l9OCT+HSD/tf27LbbhH4ZbxtHefuGug5OG7FVcR0Q/jbaZqNKk2UrQX/mB0c9p83w7Mm8LmyJ5gaZjs2CHjaxpzm2oAw+PHHK/IBU/ufxtfE0LIiGxl+b3epjCesUEzB/Luv1itgJQEMKIPH+z0TUqYSyc0bBUpeuVy65AWGM1PxEWtgDKIbKlnHjsilT56ttFcBrqjM1hFUSpBrtvFgVnKRtWfSUsXxVZE9AHs76sbCYfzDECqnOhRBrfW4ncirKtMuZ990yrhgq+Y64GKrCeg1MH7aI79WnGnfgIoVgoIG0o4TtfS9MDfG8wwCteDlRpXHMozzGc0nKYUScX3qFDFZArRtAqs4JulFwkI7iksm2mWxIEk1JEF+CGb1TL6wUz5sOAHYzZ3ZpRfYYxQKA4Zsa8vrdk+cA3XQGbMjJvGyo6CcrlQABVgmVpBodDE3CoMphctOsVjBL0j38NOtfbgwcSWYJS+4VLKi9pXKPAvIHHib3LoT9Rveh/uG1ALlKFF0CcBzC7XrkLa3uULyCdKWGfI1loBBXSVCj32h8ZkwmNQVjm7ocf9WPmocpDRfol6f/TS8GfknBjK+ffv57OZm9P7ZutRN/plybdzD1XUQo5D6weXfgFIakkAbHC4vRt5E+HoBbhl7t4xvvot4uS4mZMkpGaYpOcNaivzn93+TX/CVw80Dd9qvylZ57F4y38KcPa+Z+yzlmc+gpomuNQR8vMTrsSwAN8EUk8vmmp7MMJ74BR7mwGFIOO+ubutXUwLX9AZaPpDWDxsridAoTAncfuUB+RnWwC1EGLDbJzqkuqwVCTDGewZ0b7LomluhYjdc+UkNszz3+Hd70SiDg1GSCYX1aacDCsQxUn0c4/NJEMc55UUcB1YF99qr1vYpY+sFtvU6Y0IYiGGttbx7Bfls+52mR+qh1/etxl/aje+gsf24Z8D5aTj+AcG58fBm3/m6nf8CUEsDBBQAAAAIAAAAcV3kcdlKgxEAAKYsAAAQAAAAZmxhc2hfcnVudGltZS5weaVa61bbSLb+n6eoSf+QnMgyNteYVq9FwEnoJsCAk1lzGJaWLJVtBVlSqyTAi8VzzAPNi8236yLLF+icc9yrg1yqqr1rX759Kb99+/aqSlk55SyP05RHbMiD2dHp39mnJBDT79dXjKeTOOUOK3gZxGmcTlhcChanY17wNOQsnuUJn/G0DMo4S923b9++wVBWlCwQpXkMs3xunqfYOIlH5usPkaXmORNvxkU2Y3lQ0hSmhy/x1UwR1SgvspALUY/M68dyWvAgAo/1QDzj5rkqEuzpFvzPioOzNzilf3zx9evpkHnM6m3vbx9sj0dbBzujvf1u2OWj3V5vJxod9HrRwcE4DPbG4cE4sOS6j2cXH2lVd59/6EXjD7vbu3xrtB3tR9sHezvb3f1uj+8E+7sfut3tcH8vUqu+XZ1h0dialmUu+p1OETy4k7icVqNK8CLM0hJidMNs1tFa6BxnSTBqp1nJR1l2JzpPC6afO3fBZJJwf/F2TDq7F0X7Po541q5yEQYJL9w4n6cj682ns6PrL40jj3Z3DoJeGPbCD70P0Xa0s7e/N+592N8ORry3f7C/x8OdiPN9682biI8ZfyyLICx9ZRA2mG/13zB8MFgFCXbUmnXFNOjaI2uUZCNmvRdlYSc8lQtaLmwmi7jdej+y/rVlvadBd8ofo3gCpdgtuWE8Nnv+zWNG3IoWfYogFpzBbkm7g6LICtsyZiuyqiCjjCDJuJyzKB7DUAWTZkVmXvD7mD/A0I3YXHbFx5Ugwy4zVsAb4tK1FCPpCKciA3WTLIiEPIJ8Mc4KFvIkgR9g0o1Fz8K6XfCII9CYeuOX85xbt3Qai05vLebRhxQfpxWvB0uImhRkuT+yOLXVRupk1m2rnkZMpNiOmICvuXlQCG7T4pY7yqL5MhVwFOOUogzgtTatc+SqIyHiSdpiQRrh/7ndnKRmnAczrt6nbhx5nnX17dy/Pr46vRzCmk4GluJEygLbumVQTHgpWsv0peZ4WRWK2SQueREkPr8PEsmNi4eKq9Nt0vClwiejNjI2fGEpYOeer8IVKVAabY1TvjEJWwlSc/cLG9zzYs7GVRoSfnVCbAR4ExrveORgjzCpCFXYHS9SnggHYJnzRKKhw8g06EHvJ3IcDfqcdAo+ChIIUs6a8SgO2OkF1hbZpAB+OQzemYyC8E4o4VYzXsQYY5MqKCLhKhOEaASM4SZdSHmhbH0Wqe5VDTvMpomf9MlO+NgofJ6Ga6PHdG58a7Vu36yoKqpmudzqaxZVCbeJmCf5chhZtg/7yXAi7+a2ZaTF/aAsi3hUlRj/FCQCmn2jZHuJqSZcgGBQJaWQ259jy2ERpAIHhSi0hhBQjqdBOuEsSxMcMo3LGDL6dsrCaRYjEByylEOHbJbB2ecsqHXJyEmJpAxKtBdZxD1kVPqN89uCJxACHajVN+embytLVuX2s+uMZH+eDjnk0uzakVK4IlnDE3niZrdbN4Pah5+b8KS3wl7W9en/DPzjLxenx4Nr63kFnJSQQfMvvXZlCSLtLLvnttV7ZDoUWa3D+i045EVpbzlsacLSTgsKmr6y+oLnha33gcVB8dyziCVLeULzmCuYp3bTJg/3WwEp2HRcAGU8Jm22lpPnPQG5ZyDP/XBkPTNOb7s7yxOKsS8S4ExhJpxnKV8FYUMiJvgq5ZR1oBxnVVpzseFtwe4easXLI7l3fP6QATbWN9OE7x5cWAhp3JIrrD4NyUdPiiOTgiptzSF0JdnwhkW1zgMdHOzLGf2NgP0tR+RHWCZXjZSfQ/nkyRGrUv6Y87DkUTJ3G0rXXkG27054SnjoS6+QytOYnkP/pAidiCzhOSiSrWzEmJartlqDT0Wfhsfxoz+LBWUDfpKFcr2waVMdmLigYyhjrFIdcen1e+tfqWWylw1RR61sUQ7wVzFJEtog0HMV6hapt5FmVBUUnSDnIApynYhrqZLVxgnX9OFsyHB8kyi6+dzCEH/kodVqwr6arcQtyiz3sweEQhsGdqeZ1El1LqoyTvTSkMc5CYfydTW3YykltXXO7lIyVYuJDEivcvljLEhHBhnNljDplRxML6BM36dkx9b6K4tGxkMGkhIvikH3UtG31Y43Vh5HzVQKUpoF0uPUQjecRQkZ12KKMVjMoYzWbh5yVaYtcCeyBODXajXByGwO3kvuk2rt1t88w5MaBl8UuGpyJCR4uuZw2bs35kmU4WaVYFrka7nwIRJc9hDQBDhJcc+jpgeG0ziJwGVDFHqEZFcVAiYoEWE5D62Jke0gGgtYi1n4/kbtdLsCtqQws9BFXJnFSOaaIldiD3lu7AwpwnUVTrUysRiRtZ7tA9UTcpBa6Q8BwikREOu8OLI2zKrS6269eBS53ytM38VJ8n/md51JSW6dsde2bBg8tpb+Gk55eIf1FCYzLXJ6NI4ph5suKEc6FpWisNzaERugnGfu7C6KC1sJT0gDcJic6md3K/awKNFdOIZ9Y9HGjkWZG/60/7QcciDJxq0juf25DdrHjZWORelFyTEeRBH+zYoYSIMHU12rylqW0xc5T09nAd5PzoJRxxQL/mVSCXnsdT6mgBdy9Z9k5b4tYwFefRkcndB+AOIKQQqKyKFLKTBCqwYJiJ+ouArwqDDs/y9FMOZlOKXhdgQLoaduQxLNev//I2hjUJpQGUianwbD4y++Oe3K5iiO9OFEGWEp/hRxTmDXZOr1cDcg66K4Vtd2hpEFkq1BGMg2DiUXaBW8dDrkPGUl5NkQ0VBrB9KK2m0wU6A441F7jAAqUPHgoAsVmiO9fogF71SoU6WP7QWj3CJhKAnL1UNIFx7HaeRHQRkIXtrIQoIkQwlZZFmpyRFOIRzEYy5kdFCOTe/dYpJkI9syb1XEbXC5FCa1yBrhtV7XjK/IxUrbAsKgvrCgRM3SjRm67a/1MOijMuOn3KUHha3EbE1CBxjN8TurJWOkGwufJG436hXNJxJ2O7ixaDtEScp/ZRFDOwcySK7y1errhGaFZiOqS2y1L66lwpzvlA7Lx9bymTYp96gkX5BtpYvzgUy1El5y9vvvlwujpeokYSdKmS675jz6ftXrfNnWb3LVACi4tmpwd0iNDfU2QuIlFYOkIyjKho0o257kVY3Vv7Dz71/P2J8VNTSoWO5juUR4lVmYpOj428mRNEUoV7ARsn2YJh1C905UNkWG/rIrpfdxFAdtMYulr0iabTDjkUacGaC5mLtjJMbmucygGsvZWJwsf7AdNQCC0gvFvZNmBCO8wEOFACKWfFBpJnsQ3g1la67svdjgqCVtgsbILGiBekcjkBfZklqgfPjWYIfsU2K/1q+91/36midIz9jny29suMP+8++ei0qOPfB4MoVMHzipU6sOORzxIp8MTBF3EfK0kDsgRzxy2f6h/Ifo3/R7tytlvyOlKcXoYcrN1q3h3hlj85LW3XRvm996jQw3R4lQ2mOLOH5SpJ/77Il2fj5kT7R5p7vV2+m7vfEz+xx/ZDTEOuxJklx+Z4FIJaYr0QTyozW/0tStv05Ur1FXSLPLEuiXfdnuaN9A+YcKLEsBgwU8lgRbpQWAmdJhk51RAkvpV8MjRlUEnPJh64Azu8qJVrPdJj2oyv0knsWohH5kI/VoKppxPZeK8+XCXMOINZAuYraS8MFJw3QOoqeZcC3aTrHw2xLdlcZAg6gG2ZCKPEv1W5UqlydADjmya8vZarV+q8+wzuk32VGh+LnEZM6LdpjE+TK3zdpP8iYFSjlBoXTmMCp1HODoo09EBWqWNBLezsGWGjSHNC+297ZMrKLWu6yOfCpioFddO34+/YiUvcxmKPIpAjnUG8oE90UIRwinUmMT7kskFC9VnU3+6hRXDbacRikqZ9PTz6ayq3WvKqcbqKsGVMhZrlENAx0diHQQXq9Zf2HHtCGkQJGEbtGAxwAHgi0cPklEh2QALR426n7V71yEBg07CrjjFAjtUy7gbUwjLH1v1JETTZcA1KlwWlaALaoZxdvRvOQA3tfi7Pudd6TO1+D9pqZc8lkOkO6U6o8eJSHjoAB4Zv34kbfr2yyl/9djxw3xSVkDli84br3C8q3R8sLEahktZOhIyZjeiCyi5IjOIzrLnMrLnYWd1EWY1vU1pUFBjkCkOksy9oM8wr8wF690HUU5IvtDSsUSDGpmQ2Sm0w4F7s6wiEsYmtzSrPeQ2xZBFGeet+v2PrhbVAZVE7KbcRByH3XQzXTsP/Ly9teufCsC1A48FVkhfvO23F0M8TjNcvntgKo1lEs8ztrj8SznExrds27rPotxxPC+1yxRT+WoxPd+zZyLE/M0sq0Mf0IUSvNymqVtCutIq0Fwx+1uaTtcLUjEXLjUnarKYJRwZBozsJbHuSwnpYPo9CPmSO3fGZLrBUllmpGexICOVeJLEP9Z685ckmpQpqzJrNlQEj+gtly5UHbxlU5o65tex5TxAELSLxA8z1LBVwPjg2feSHiwe+8ofr5712u2ntZuXsmcAAJ04aEBg8yS0F72jkhidA1udqiP8gDr4b50j8VtpioY5RXuCql6nUQutUxDl5pBzZbldqzZTCs0LFRPUEl9tVF2qCdoviQ0qn10T3YRHWy1A8zjnqfUz1eg6jwZVcroggqvv7gjd+p3dBet3tBd8jqUWIarepdmoepY6prRX7RuA4SLKtUdWKsvo8j6rkYYWji+mAa93T2r37gpx/daZvXt+NKd+IZ9X99O5/EbN3s22ruHUqI4LO0MLoZ0vkAAlmemxsjV4PLCWxTJ9fjXi5PB2bV8o4Bx8eofF1d/+CenV/KlDJhrfH+9PPt4dPzH4PzEs44mE8v5fHV0cnrhH50fnf1zeHp87Q/Ojz6eDfBaXnxYzuU/h18uzv9xdHV+ev752rPUTaNVn8HNsxxFtuFhcP759HzgDwfXQ8uhPEZNVLkvEk6kKOQi5scCK3fH1HoWslvKPl1290xYdVj5kLUpb9a3vkgX5F0Q8v+Ik4gLdz0h1jRP9XWlvq3qs2FVjDKk1b3//Bv/plToJKj1ZDjosOOrT3Sr1CHfno9hY3hU904xjJ1dfPr0MqlvQlVxpxS8mPwJCPyeMgQ2yh4depcylRVy9p1eU6wB9RzmI7NXyEZdRdNh6+/AFkmBMu0XaI+tIWWTMhPFGZ82JITPwiCGyZ2J0NNKPolZSFAZJaguG045+CL6lIrp3R0JzSJHRkuXu/FMbOBKwrMGDO0MCOtIwCRAW4HljCqqtXFAryvRGa8bvVSEoGbteymXrQejSjWSFIRRL/Mh8mSaDMv08L+jml9LW51eDmiYF0Vz+Hp4cvFt6EjR+Cl/8HUJtFJhrcPhpusVQCJdb/Rp0MWTU98s9FduQ8yM1vKVxPOCYgQn8uqfVbkDYG/Z6HjrpoRH61yk9hlSkzhsTKCFc+9GuvNtPZoQegrOU+9G77B4R4XHA2WfvipO7JVbg4cpsjxpBMSbbJ/bPVUuqeNk1JNfr+AaypXVlLd2OUufte5YTfaViC9bzv1Op9vbd7fwX7e/f7C31aG4Y6hZdTrQeyUb2MhnXVfYZlFr4yIp6put2833tUtip1kvqsx8XmuNvXTHbHh4+ViYtEq53eTrt+2tlxcruUi0O1MdVAUpdd0+4iHl2NQuQHgiN5UVwPaWOJQgkssMiWo6s7TMaHVJzRwCT1MQb2xwbPpsrBJf+jSuNjd9NjZR1TpqaXjLDY41MWpXcuqOxwYYdlYAd51fpUUQ2KwGJX41473Frmuh0o9zymlcwzw5JP1SVEnb3L8d1l3SSIUoFWDMj61+QvA/JfAVQS8AbCifbPVbGa+JM04U8JlBXFeKreEUa7igYIgmr8uJSm4vEyqfl6Ckb0GoQE0zZHR7u7vbextlL4EN6/sjrL5bxyfqfdJ7V6UeNieHFJ6FRC1BqWe1DhHKVDqtfgp4qDRGz4hLkWe9LGLVjAoVprb6r95nSG/iEdyKc/0jz/V427hYpfQ2WZGhBHDqTSyzsabh/wJQSwMEFAAAAAgAAABxXVGBE88YAgAAyAcAAAsAAABhc3NldHMuanNvbsWU244TMQyG7/cpql6vOjkn3lsQDwCIG4Qq20nagdIpk+lKq9W+O5muWgT0JIHUm5ES/07i7x/7+W4ymfbpsS1tt54+TKbGkQerBBiH7HxMDCZK65wB4UCA1Vw3iaf3YyaWkoZS8z7X1WTyvPvW/Q0Oy/G0YdtT18R2mBXMaUjr0vVll7qTrfF7Osjm85M6ehrSeIt13gkvNKhDqCxRWTce4oNTBEr6IL3LqLG+Uht2LLyMyOjIOiQZY6SQfM6AQrLx2SrlEgf6dV3ptj3vHrYchk15aJrldrFo14uMnGbcNf2Wl7hePGHXvFthWX768H6+WW1L06fSrR5Tcy3F5gKg3K7SaY471cv9OfSrH/NN332dt+urHLgk3xshpZU6GOWPGZG8J+0DJqm0USllIhsyKFPjWTuFNZlAc/I6Rq2CCJlHg4CFlQbcDY24VP/vfpxQn7WFsKTLDTGq/rUfbIjOsMySNFoLhJkcEAlpskZS3tgUhOAARlvMkaWt7F0icMQ1CDex4TyePf6jqsvYr22GV/r/pRdQ5gTgs6x1ClkHEYUYTbSBtRgL9nWZmbX15LSpkwuCB4U6oLYuhng7E65thXPis5Z8fPM2cRdTP+Nvm+FvF07FD9wDCBm01sdGkFKGImdVp3v9l5myiRp0zlZ7w4pQQAKhgo02jtaoqCg7TSAtOenkTbCfKnhP+o/4K9z6/XL3cvcTUEsBAhQDFAAAAAgAAABxXSN7CadvCwAAFyIAAA8AAAAAAAAAAAAAAIABAAAAAGZsYXNoX2NvbW1vbi5weVBLAQIUAxQAAAAIAAAAcV2R6Gn5LgMAAIkHAAAWAAAAAAAAAAAAAACAAZwLAABmbGFzaF9jaGVja19kYXRhc2V0LnB5UEsBAhQDFAAAAAgAAABxXfaskhqyCQAASRsAABAAAAAAAAAAAAAAAIAB/g4AAGZsYXNoX2RhdGFzZXQucHlQSwECFAMUAAAACAAAAHFd5HHZSoMRAACmLAAAEAAAAAAAAAAAAAAAgAHeGAAAZmxhc2hfcnVudGltZS5weVBLAQIUAxQAAAAIAAAAcV1RgRPPGAIAAMgHAAALAAAAAAAAAAAAAACAAY8qAABhc3NldHMuanNvblBLBQYAAAAABQAFADYBAADQLAAAAAA='))) as archive:
    for name in archive.namelist():
        assert Path(name).name==name
        (RUNTIME/name).write_bytes(archive.read(name))
sys.path.insert(0,str(RUNTIME))
importlib.invalidate_caches()
for name in ('flash_runtime','flash_common','flash_dataset','flash_check_dataset'):
    old=sys.modules.get(name)
    if old is not None and Path(getattr(old,'__file__','')).parent==RUNTIME:sys.modules.pop(name,None)
from flash_runtime import run
run(RUNTIME,WORK,MAX_JOB_SECONDS,MAX_STARTUP_SECONDS)
